In [ ]:
import math
import time
from enum import Enum, auto
# utils
from utils import *
from draw_utils import *

import cv2
import matplotlib.pyplot as plt
import numpy as np
np.set_printoptions(precision=4)

from filterpy.common import Saver 

# hand landmark: https://google.github.io/mediapipe/solutions/hands.html
import mediapipe as mp 
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

# how to control your mouse: https://stackoverflow.com/questions/281133/how-to-control-the-mouse-in-mac-using-python
# mouse, keyboard controll: https://pypi.org/project/pynput/
from pynput import mouse as Mouse
from pynput import keyboard as Keyboard

# Mac screen info
from AppKit import NSScreen
SCREEN_INDEX = 0
SCREEN_WIDTH, SCREEN_HIGHT = NSScreen.screens()[SCREEN_INDEX].frame().size.width, NSScreen.screens()[SCREEN_INDEX].frame().size.height
print(f'screen size: {(SCREEN_WIDTH, SCREEN_HIGHT)}')


In [1]:
def preprocess_img(image):
  h, w = image.shape[:2]
  # resize
  if h < w:
    img = cv2.resize(image, (DESIRED_WIDTH, math.floor(h/(w/DESIRED_WIDTH))))
  else:
    img = cv2.resize(image, (math.floor(w/(h/DESIRED_HEIGHT)), DESIRED_HEIGHT))

  return img


In [ ]:
def landmarks2vec(hand_landmarks):
  vecs = np.empty((21, 3))
  for landmark_idx in mp_hands.HandLandmark:
    vecs[landmark_idx] = np.array([
      hand_landmarks.landmark[landmark_idx].x,
      hand_landmarks.landmark[landmark_idx].y,
      hand_landmarks.landmark[landmark_idx].z,
    ])
  return vecs

def switch_hand(landmarks):
  landmarks = np.array(landmarks)
  landmarks[[1, 2, 3, 4, 5, 6, 7, 8]] = landmarks[[17, 18, 19, 20, 13, 14, 15, 16]] 
  return landmarks 

In [ ]:
def shift_to_palm_coordinate(landmarks, is_right_hand):
  """
  origin: wrist
  y-axis: index-mcp to wrist vector
  z-axis: orthogonal to y-axis vector and wrist pinky-mcp vector, point out of palm 
  @return shifted_landmarks, rotation_matrix
  """
  landmarks = np.array(landmarks)

  # shift origin
  delta_origin = np.copy(landmarks[0])
  landmarks -= delta_origin 

  # calculate rotation martix
  axis_y = landmarks[5] - landmarks[0]
  axis_z = np.cross(landmarks[9] - landmarks[0], axis_y)
  # correct z-axis direction
  axis_z = axis_z if is_right_hand else -axis_z
  r = get_coordinate_rotation_matrix([-axis_y, -axis_z], [[0,1,0], [0,0,1]])
  
  return landmarks @ r, r,delta_origin 


def shift_back_to_origin_coordinate(landmarks, rotation_matrix, delta_origin):
  landmarks = np.array(landmarks)

  # rotate back
  # NOTE: invsere of rotation martix = transpose of rotation matrix
  landmarks = landmarks @ rotation_matrix.T
  
  # shift origin
  landmarks += delta_origin
  
  return landmarks

def annotate_3axis(img, rotation_matrix, delta_origin):
    start = shift_back_to_origin_coordinate(
        [0, 0, 0], rotation_matrix, delta_origin)
    x_end = shift_back_to_origin_coordinate(
        [40, 0, 0], rotation_matrix, delta_origin)
    y_end = shift_back_to_origin_coordinate(
        [0, 40, 0], rotation_matrix, delta_origin)
    z_end = shift_back_to_origin_coordinate(
        [0, 0, 40], rotation_matrix, delta_origin)
    cv2.arrowedLine(img, (start[0:2]).astype(
        int), (x_end[0:2]).astype(int), thickness=2, color=(0, 0, 255))
    cv2.arrowedLine(img, (start[0:2]).astype(
        int), (y_end[0:2]).astype(int), thickness=2, color=(0, 255, 0))
    cv2.arrowedLine(img, (start[0:2]).astype(
        int), (z_end[0:2]).astype(int), thickness=2, color=(255, 0, 0))
    cv2.putText(img, 'x', org=((x_end[0:2] + [5, 5]).astype(int)), color=(
        0, 0, 255), fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=.7, lineType=2)
    cv2.putText(img, 'y', org=((y_end[0:2] + [5, 5]).astype(int)), color=(
        0, 255, 0), fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=.7, lineType=2)
    cv2.putText(img, 'z', org=((z_end[0:2] + [5, 5]).astype(int)), color=(
        255, 0, 0), fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=.7, lineType=2)


In [ ]:
class FINGER_STATE(Enum):
  BENT = 1
  STRAIGHT = 0
  UNKNOWN = auto()

class FINGER(Enum):
  THUMB = 0
  INDEX = 1
  MIDDLE = 2
  RING = 3
  PINKY = 4


def get_finger_state(landmarks):
  # TODO: detect thumb state
  res = np.array([FINGER_STATE.UNKNOWN for i in range(5)])
  
  # finger_idx
  # thumb: 1~4, index: 5~8, middle: 9~12, ring: 13~16, pinky: 17~20
  for finger_idx in range(1, 21, 4):
    tip = landmarks[finger_idx+3, :, 0]
    dip = landmarks[finger_idx+2, :, 0]
    pip = landmarks[finger_idx+1, :, 0]
    mcp = landmarks[finger_idx, :, 0]

    accumulated_angle = (np.pi - angle_between_vectors(pip - mcp, pip - dip)) + (np.pi - angle_between_vectors(dip - pip, dip - tip))
    
    if finger_idx == 0:
      # NOTE: special case -> thumb
      res[finger_idx] = FINGER_STATE.BENT if accumulated_angle > (np.pi * 0.25) else FINGER_STATE.STRAIGHT
    else :
      res[(finger_idx-1) // 4] = FINGER_STATE.BENT if accumulated_angle > (np.pi * 0.4) else FINGER_STATE.STRAIGHT

  return res

In [ ]:
DEBUG_MOUSE = False
DEBUG_KEYBOARD = False 

DESIRED_HEIGHT = 720
DESIRED_WIDTH = 720

landmarks_Q_ratio = np.array(([
  [0.3284, 0.1398, 0.7641],
  [0.4456, 0.1646, 0.8197],
  [0.597 , 0.1817, 0.8386],
  [0.7696, 0.1642, 0.8617],
  [1.    , 0.176 , 0.9134],
  [0.5905, 0.1456, 0.8833],
  [0.7556, 0.2004, 0.8921],
  [0.8759, 0.5362, 0.916 ],
  [0.9902, 0.959 , 0.9806],
  [0.5506, 0.1247, 0.8535],
  [0.6859, 0.2006, 0.9302],
  [0.7351, 0.5873, 0.9531],
  [0.7719, 1.    , 0.9764],
  [0.4939, 0.1114, 0.8318],
  [0.6125, 0.194 , 0.9084],
  [0.6481, 0.4704, 0.917 ],
  [0.6687, 0.756 , 0.9214],
  [0.4182, 0.1131, 0.8288],
  [0.5076, 0.1615, 0.8896],
  [0.5515, 0.2703, 0.9429],
  [0.5887, 0.4023, 1.    ],
]))


In [ ]:
mouse = Mouse.Controller()
keyboard = Keyboard.Controller()
Key = Keyboard.Key
if DEBUG_MOUSE:
  mouse_listener = Mouse.Listener(
    # on_move=lambda x,y: print(f'Pointer move to ({x :.2f}, {y :.2f})'),
    # on_scroll=lambda x,y,dx,dy: print(f'Scrolled ({x :.2f}, {y :.2f}) at {"down" if dy < 0 else "up"}'),
    on_click = lambda x, y, button, pressed: print(f'{"Pressed" if pressed else "Released"} at ({x :.2f}, {y :.2f})')
  )
  mouse_listener.start()
  mouse_listener.wait()
if DEBUG_KEYBOARD:
  def on_press(key):
    try:
        print(f'alphanumeric key {key.char} pressed')
    except AttributeError:
        print(f'special key {key} pressed')
  def on_release(key):
    print(f'{key} released')
    if key == Key.esc:
        return False
  keyboard_listener = Keyboard.Listener(on_press=on_press, on_release=on_release)
  keyboard_listener.start()
  keyboard_listener.wait()

# fps
s_time = time.time()
frame_cnt = 0
prev_frame_cnt = 0
prev_timestamp = time.time()

# init estimate
existence = [.5, 0]
handedness = [.5, 0]
landmarks = np.full((21, 3, 2), 0.5) # TODO: is 0.5 a reasonable init value?

# filter
dt = 1./15
existence_f = pos_vel_filter(existence, P=1., R=.4, Q=.5, dt=1.)
handedness_f = pos_vel_filter(handedness, P=1., R=.3, Q=.5, dt=1.)

pos_Q = np.tile([.25, .25, 1.25], [21, 1])
vel_Q = landmarks_Q_ratio * [52.5, 54., 12.5]
ls_Q = np.eye(126) * np.stack([pos_Q, vel_Q], axis=2).flatten()
x_R = .25
y_R = .25
z_R = .75 
ls_R_block = np.eye(3) * [x_R, y_R, z_R]
ls_R = block_diagonal_array(63//3, ls_R_block)
landmarks_f = multi_pos_vel_filter(landmarks.flatten(), P=DESIRED_WIDTH, R=ls_R, Q=ls_Q, dt=dt)

existence_s = Saver(existence_f)
handedness_s = Saver(handedness_f)
landmarks_s = Saver(landmarks_f)

# gesture
click_queue = np.ones(4)

# DEBUG: params
tmax = -100


cap = cv2.VideoCapture(0)
with mp_hands.Hands(
    min_detection_confidence=0.75,
    min_tracking_confidence=0.7) as hands:
  while cap.isOpened():
    success, raw_image = cap.read()
    if not success:
      print("Ignoring empty camera frame.")
      # If loading a video, use 'break' instead of 'continue'.
      continue

    # Flip the image horizontally for a later selfie-view display, and convert the BGR image to RGB.
    image = cv2.cvtColor(cv2.flip(preprocess_img(raw_image), 1), cv2.COLOR_BGR2RGB)
    # To improve performance, optionally mark the image as not writeable to pass by reference.
    # NOTE: see no different
    image.flags.writeable = False
    results = hands.process(image)
    
    image_hight, image_width, _ = image.shape
    # Draw the hand annotations on the image.
    # NOTE: see no different
    # image.flags.writeable = True
    annotated_image = image


    # existence detection
    existence_f.predict()
    existence_f.update(bool(results.multi_hand_landmarks))
    existence = existence_f.x
    # DEBUG: save existence
    # existence_s.save()


    if existence[0] > .5:

      # handedness
      handedness_f.predict()
      # NOTE: `existence_f` only response for one hand existence
      if existence_f.z == True:
        raw_handedness = results.multi_handedness[0]
        z_handedness = .5 + (.5 if raw_handedness.classification[0].index == 1 else -.5) * raw_handedness.classification[0].score 
        handedness_f.update(z_handedness)
      handedness = handedness_f.x
      # DEBUG: save handness
      # handedness_s.save()


      # landmarks
      landmarks_f.predict()
      if existence_f.z == True:
        raw_landmarks = results.multi_hand_landmarks[0]

        # preprocess z_landmarks
        z_landmarks = landmarks2vec(raw_landmarks)

        # switch hand landmarks
        should_switch_hand = (handedness[0] > .5 and handedness_f.z < .5) or (handedness[0] < .5 and handedness_f.z > .5)
        if should_switch_hand: 
          z_landmarks = switch_hand(z_landmarks)

        # scale to image size
        # FUTURE: may have a better scale factor
        z_landmarks *= [image_width, image_hight, 1] 

        # FUTURE: change coordinate
        # z_shift_landmarks, rotation_matrix, delta_origin = shift_to_palm_coordinate(z_landmarks, handedness[0] > 0.5) 
        # landmarks_f.update(z_shift_landmarks.flatten())
        
        # update landmarks
        landmarks_f.update(z_landmarks.flatten())
        
        # DEBUG: save landmarks
        landmarks_s.save()

      landmarks = landmarks_f.x.reshape(21, 3, 2)


      # Gesture
      finger_states = get_finger_state(landmarks)
      # FIXME: adjust distance of diff depth to the same scale
      ti = np.linalg.norm(landmarks[4, :, 0] - landmarks[6, :, 0]) > 23.
      im = np.linalg.norm(landmarks[8, :, 0] - landmarks[12, :, 0]) > 40.
      mr = np.linalg.norm(landmarks[12, :, 0] - landmarks[16, :, 0]) > 40.
      rp = np.linalg.norm(landmarks[16, :, 0] - landmarks[20, :, 0]) > 40.
      
      # mouse click / drag
      ## prev
      # DEV: `ti` cannot separate idle and click clearly 
      # if ti:
      if finger_states[0] == FINGER_STATE.STRAIGHT:
        if np.all(click_queue == FINGER_STATE.BENT.value):
          print('release!')
          mouse.release(Mouse.Button.left)
          draw_click_drag(annotated_image, landmarks, is_click=False, is_drag=False)
        elif np.all(click_queue[:-1] == FINGER_STATE.BENT.value) or np.all(click_queue[:-2] == FINGER_STATE.BENT.value) or np.all(click_queue[:-3] == FINGER_STATE.BENT.value):
          # FUTURE: current condition cannot support "double click"
          print('click!')
          mouse.click(Mouse.Button.left)
          draw_click_drag(annotated_image, landmarks, is_click=True, is_drag=False)
      # DEV: `ti` cannot separate idle and click clearly
      # elif not ti: 
      elif finger_states[0] == FINGER_STATE.BENT:
        if np.all(click_queue[:-1] == FINGER_STATE.BENT.value):
          if click_queue[-1] == FINGER_STATE.STRAIGHT.value:
            print('press!')
            mouse.press(Mouse.Button.left)
          draw_click_drag(annotated_image, landmarks, is_click=False, is_drag=True)
      ## update
      if finger_states[0].value != FINGER_STATE.UNKNOWN:
        click_queue = np.roll(click_queue, 1)
        # DEV: `ti` cannot separate idel and click clearly
        # click_queue[0] = FINGER_STATE.BENT.value if ti else FINGER_STATE.STRAIGHT.value
        click_queue[0] = finger_states[0].value

      # move mouse
      # DEV:
      if im:
      # if np.all(finger_states[[2,3]] == FINGER_STATE.BENT) and (
            # (np.all(np.abs(landmarks[0, 0:2, 1]) < 10.))
            # or 
            # (finger_states[1] == FINGER_STATE.STRAIGHT)
          # ):

        max_move_speed = 100
        x = np.clip(landmarks[8, 0, 1] , -max_move_speed, max_move_speed)
        y = np.clip(landmarks[8, 1, 1] , -max_move_speed, max_move_speed)

        # WARN: 
        # mouse may move to the negative position which used to refer to second monitor, 
        # but it also cause mouse move to non-monitor area, so I clip mouse position to 
        # keep it in the monitor area.
        # FUTURE: support multi-monitor
        move_x = np.clip(x, 0 - mouse.position[0], SCREEN_WIDTH - mouse.position[0])
        move_y = np.clip(y, 0 - mouse.position[1], SCREEN_HIGHT - mouse.position[1])
        mouse.move(move_x, move_y)

      # scroll vertically
      # TODO: smooth scroll (keep scroll after gesture disappear)
      max_scroll_speed = 5
      scale_factor = .07

      ## scroll down
      # DEV:
      if mr: 
      # if np.all(finger_states[[3,4]] == FINGER_STATE.BENT) and \
            # np.all(finger_states[[1,2]] == FINGER_STATE.STRAIGHT):
        scroll_y = np.clip(landmarks[8, 1, 1]*scale_factor, -max_scroll_speed, 0)
        mouse.scroll(0, scroll_y)
        # annotate
        cv2.arrowedLine(annotated_image, landmarks[8, 0:2, 0].astype(int), (landmarks[8, 0:2, 0] + [0, -scroll_y*10]).astype(int), color=(255, 50, 50), thickness=3)

      ## scroll up 
      if rp:
      # if np.all(finger_states[[4]] == FINGER_STATE.BENT) and \
            # np.all(finger_states[[1,2,3]] == FINGER_STATE.STRAIGHT):
        scroll_y = np.clip(-landmarks[8, 1, 1]*scale_factor, 0, max_scroll_speed)
        mouse.scroll(0, scroll_y)
        # annotate
        cv2.arrowedLine(annotated_image, landmarks[8, 0:2, 0].astype(int), (landmarks[8, 0:2, 0] + [0, -scroll_y*10]).astype(int), color=(255, 50, 50), thickness=3)


      # switch desktop
      if np.all(finger_states[[1,2,3,4]] == FINGER_STATE.STRAIGHT) and \
        finger_states[0] == FINGER_STATE.BENT:
        pass
        # if landmarks[8, 0, 1] > 300:
          # NOTE: the following 2 method work as cmd+left/right, not ctrl+left/right 
          # with keyboard.pressed(Key.ctrl):
            # keyboard.press(Key.left)
            # keyboard.release(Key.left)
          # keyboard.press(Keyboard.KeyCode.from_vk(59))
          # keyboard.press(Keyboard.KeyCode.from_vk(123))
          # keyboard.release(Keyboard.KeyCode.from_vk(123))
          # keyboard.release(Keyboard.KeyCode.from_vk(59))
          # NOTE the following method work in terminal  
          # osascript -e "tell application \"System Events\" to key code 123 using control down"
          

      # Control Center
      # App Expose
      # LaunchPad
      # Show Desktop


      # Draw 
      # measurement landmarks
      # mp_drawing.draw_landmarks(annotated_image, raw_landmarks, mp_hands.HAND_CONNECTIONS)
      # draw_landmarks(annotated_image, np.stack([z_landmarks, np.zeros(z_landmarks.shape)], axis=2), (255, 50, 50, 0.5))

      # Draw normal landmark
      draw_landmarks(annotated_image, landmarks)

      # FUTURE: Draw projected landmarks
      # inv_landmarks = np.stack([
        # shift_back_to_origin_coordinate(landmarks[:, :, 0], rotation_matrix, delta_origin), 
        # shift_back_to_origin_coordinate(landmarks[:, :, 1], rotation_matrix, delta_origin)], axis=2)
      # draw_landmarks(annotated_image, inv_landmarks)
      # FUTURE: Draw projection axis
      # annotate_3axis(annotated_image, rotation_matrix, delta_origin)      

      # Draw handedness
      # draw_handedness(annotated_image, handedness)

      # Draw finger states
      # draw_finger_state(annotated_image, handedness, finger_states)
      
      # Draw move click / drag
      # is_click = np.all(click_queue[:-1] == FINGER_STATE.BENT.value) or np.all(click_queue[:-2] == FINGER_STATE.BENT.value) or np.all(click_queue[:-3] == FINGER_STATE.BENT.value)
      # is_drag = np.all(click_queue== FINGER_STATE.BENT.value)
      # draw_click_drag(annotated_image, landmarks, is_click, is_drag)
      

      # DEBUG: output 
      # if time.time() - s_time > 1:
        # tmax = np.max([tmax, np.abs(landmarks_s.z[-1].reshape(21,3)[8, 2] - landmarks_s.z[-2].reshape(21,3)[8, 2]) / dt])
      # tmax = np.max([tmax, np.abs(landmarks_f.x.reshape(21,3,2)[8,2,1])])
        # cv2.putText(annotated_image, 
            #  f'{landmarks_f.x.reshape(21,3,2)[8,1,1] :.2f}',
            #  f'{time.time() % 20}',
            #  org=(int(image_width*.01), int(image_hight*.15)), # bottomLeftCornerOfText
            #  fontFace=cv2.FONT_HERSHEY_SIMPLEX, 
            #  fontScale=.8,
            #  color=(255, 255, 255),
            #  lineType=2)

    else: 
      # TODO: reduce to default position and uncertainty
      # handedness_f.update()
      # landmarks_f.update()
      pass

    frame_cnt += 1
    now_timestamp = time.time()
    if now_timestamp - prev_timestamp >= 1:
      prev_timestamp, prev_frame_cnt = now_timestamp, frame_cnt
      frame_cnt = 0
    cv2.putText(annotated_image, f'{prev_frame_cnt}', org=(image_width - 30, 20), fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=0.5, color=(100, 255, 100), lineType=2)


    annotated_image = cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)
    cv2.imshow('Hands', annotated_image)
    if cv2.waitKey(20) & 0xFF == 27:
      if DEBUG_MOUSE:
      # FIXME: `Listener.stop()` failed to stop mouse listerner
        mouse_listener.stop()
      if DEBUG_KEYBOARD:
        keyboard_listener.stop()
      break

cap.release()
# cv2.destroyAllWindows()

## Testing

In [ ]:
xs = np.asarray(landmarks_s.x_post).reshape(-1,21,3,2)
ts = np.arange(xs.shape[0]) * dt

plt.figure(figsize=(30, 8))
start = 0
end = None

def dist(a, b, std=False):
  a = np.array(a) 
  b = np.array(b) 
  d = np.linalg.norm(a[:, :] - b[:, :], axis=1)
  return d
  # return (d - np.mean(d)) / np.std(d)
  
# Base Line
plt.plot(ts[start:end], [0]*len(ts[start:end]), color='black')
# plt.plot(ts[start:end], [23]*len(ts[start:end]), color='black')
# plt.plot(ts[start:end], [30]*len(ts[start:end]), color='black')

# Distance between Tips
# plt.plot(ts[start:end], dist(xs[start:end, 4, :, 0], xs[start:end, 6, :, 0]), marker='.', label='ti_1')
# plt.plot(ts[start:end], dist(xs[start:end, 8, :, 0], xs[start:end, 12, :, 0]), marker='.', label='im_1')
# plt.plot(ts[start:end], dist(xs[start:end, 12, :, 0], xs[start:end, 16, :, 0]), marker='.', label='mr_1')
# plt.plot(ts[start:end], dist(xs[start:end, 16, :, 0], xs[start:end, 20, :, 0]), marker='.', label='rp_1')

# Tips Move Speed
plt.plot(ts[start:end], xs[start:end, 16, 0, 1], marker='.', label='xs')
plt.plot(ts[start:end], xs[start:end, 16, 1, 1], marker='.', label='ys')

plt.legend(loc="best")

In [ ]:
zs = np.asanyarray(landmarks_s.z).reshape(-1, 21, 3)#[:-1]
prior = np.asanyarray(landmarks_s.x_prior).reshape(-1, 21, 3, 2)#[1:]
post = np.asanyarray(landmarks_s.x_post).reshape(-1, 21, 3, 2)
ts = np.arange(zs.shape[0]) * dt
print(ts.shape)

In [ ]:
plt.figure(figsize=(15, 15))
# yv: 100
idx = 8
xyz = 1
start = 0
end = -1

plt.subplot(611)
plt.plot(ts[start:end], zs[start:end, idx, 0], marker='.', label='z')
plt.plot(ts[start:end], prior[start:end, idx, 0, 0], marker='.', label='prior')
plt.plot(ts[start:end], post[start:end, idx, 0, 0], marker='.', label='post')

plt.xticks(np.arange(zs.shape[0]*dt, step=6*dt))
plt.legend(loc='best')
plt.title('x pos')

plt.subplot(612)
plt.plot(ts[start:end], prior[start:end, idx, 0, 1], marker='.', label='prior')
plt.plot(ts[start:end], post[start:end, idx, 0, 1], marker='.', label='post')

plt.xticks(np.arange(zs.shape[0]*dt, step=6*dt))
plt.legend(loc='best')
plt.title('x vel')

plt.subplot(613)
plt.plot(ts[start:end], zs[start:end, idx, 1], marker='.', label='z')
plt.plot(ts[start:end], prior[start:end, idx, 1, 0], marker='.', label='prior')
plt.plot(ts[start:end], post[start:end, idx, 1, 0], marker='.', label='post')

plt.xticks(np.arange(zs.shape[0]*dt, step=6*dt))
plt.legend(loc='best')
plt.title('y pos')

plt.subplot(614)
plt.plot(ts[start:end], prior[start:end, idx, 1, 1], marker='.', label='prior')
plt.plot(ts[start:end], post[start:end, idx, 1, 1], marker='.', label='post')

plt.xticks(np.arange(zs.shape[0]*dt, step=6*dt))
plt.legend(loc='best')
plt.title('y vel')

plt.subplot(615)
plt.plot(ts[start:end], zs[start:end, idx, 2], marker='.', label='z')
plt.plot(ts[start:end], prior[start:end, idx, 2, 0], marker='.', label='prior')
plt.plot(ts[start:end], post[start:end, idx, 2, 0], marker='.', label='post')

plt.xticks(np.arange(zs.shape[0]*dt, step=6*dt))
plt.legend(loc='best')
plt.title('z pos')

plt.subplot(616)
plt.plot(ts[start:end], prior[start:end, idx, 2, 1], marker='.', label='prior')
plt.plot(ts[start:end], post[start:end, idx, 2, 1], marker='.', label='post')

plt.xticks(np.arange(zs.shape[0]*dt, step=6*dt))
plt.legend(loc='best')
plt.title('z vel')

In [ ]:
zs_std = np.std(zs, axis=0)
zs_mean = np.mean(zs, axis=0)
prior_std = np.std(prior, axis=0)
prior_mean = np.mean(prior, axis=0)
post_std = np.std(post, axis=0)
post_mean = np.mean(post, axis=0)
print(zs_std.shape, prior_std.shape, post_std.shape)

In [ ]:
plt.figure(figsize=(12, 6))
# plt.errorbar(np.arange(21)+.0, np.zeros(21), yerr=zs_std[:, 0], linestyle='None', fmt='o', label='x')
# plt.errorbar(np.arange(21)+.1, np.zeros(21), yerr=zs_std[:, 1], linestyle='None', fmt='o', label='y')
# plt.errorbar(np.arange(21)+.2, np.zeros(21), yerr=zs_std[:, 2], linestyle='None', fmt='o', label='z')
plt.errorbar(np.arange(21)+.3, np.zeros(21), yerr=prior_std[:, 0, 1], linestyle='None', fmt='o', label='prior-x')
plt.errorbar(np.arange(21)+.4, np.zeros(21), yerr=prior_std[:, 1, 1], linestyle='None', fmt='o', label='prior-y')
plt.errorbar(np.arange(21)+.5, np.zeros(21), yerr=prior_std[:, 2, 1], linestyle='None', fmt='o', label='prior-z')
# plt.errorbar(np.arange(21)+.6, np.zeros(21), yerr=post_std[:, 0, 1], linestyle='None', fmt='o', label='post-x')
# plt.errorbar(np.arange(21)+.7, np.zeros(21), yerr=post_std[:, 1, 1], linestyle='None', fmt='o', label='post-y')
# plt.errorbar(np.arange(21)+.8, np.zeros(21), yerr=post_std[:, 2, 1], linestyle='None', fmt='o', label='post-z')
plt.xticks(np.arange(21))
plt.legend()
# print(np.std(zs_std[:, :], axis=0))
print(prior_std[:, :, 1].shape)

In [ ]:
plt.figure(figsize=(12,6))
np.set_printoptions(suppress=True)
print(np.round(prior_std[:, :, 1], 3))
plt.plot(np.arange(21), prior_std[:, 0, 1], marker='.', label='x')
plt.plot(np.arange(21), prior_std[:, 1, 1], marker='.', label='y')
plt.plot(np.arange(21), prior_std[:, 2, 1], marker='.', label='z')
plt.xticks(np.arange(21))
plt.legend()
plt.show()

In [ ]:
max_v = np.max(prior_std[:,:,1], axis=0)

# print(max_v)
ratio = np.round(prior_std[:,:,1]/max_v, 4)
print(ratio)

plt.figure(figsize=(10,6))
plt.plot(np.arange(21), ratio[:, 0], marker='.', label='x')
plt.plot(np.arange(21), ratio[:, 1], marker='.', label='y')
plt.plot(np.arange(21), ratio[:, 2], marker='.', label='z')
plt.xticks(np.arange(21))
plt.legend()
plt.show()

In [ ]:
# a = np.arange(12).reshape(3,4)
# b = np.linspace(0.5, 6, 12).reshape(3,4)
# print(a, b)
# print(np.stack([a,b],axis=2).flatten())
np.repeat([.2, .25, 1.2], [3, 3 ,3])